# Generacion de los diccionarios de k-meros (`lnc_kmer_dict.txt` y `mir_kmer_dict.txt`)

Este notebook **regenera desde cero** los dos diccionarios de k-meros que necesitan
`generate_csv.ipynb` y `generate_csv_filtered.ipynb`.

## Archivos de ENTRADA
- `data/sequences/seq+ss_join_<lncRNA>.txt`       -> linea 1 = secuencia del lncRNA
- `data/sequences_miRNA/seq+ss_join_<miRNA>.txt`  -> linea 1 = secuencia del miRNA (precursor)
- `data/txt_interac/mirnas_lncrnas_validated_positive_pairs.txt`  (opcional, para filtrar)
- `data/txt_interac/negative_pairs.txt`                           (opcional, para filtrar)

## Archivos de SALIDA
- `data/kmers/lnc_kmer_dict.txt`
- `data/kmers/mir_kmer_dict.txt`

Cada linea tiene el formato `nombre,v1,v2,...,v340`. El vector contiene los conteos de
todos los 1-, 2-, 3- y 4-meros (4 + 16 + 64 + 256 = 340 valores). Es exactamente el
formato que espera la funcion `load_dict()` de los notebooks de generacion de CSV.

> **Nota sobre los miRNA:** el archivo de secuencia contiene el *precursor* completo
> (minusculas = flancos, mayusculas = miRNA maduro). El parametro `MIRNA_REGION`
> permite usar el precursor completo o solo la region madura.


In [8]:
import os
import itertools

# ====================== CONFIGURACION ======================
# Rutas relativas a la carpeta src/ (igual que el resto de notebooks del proyecto)
SEQ_LNC_DIR = "./../data/sequences"        # secuencias de lncRNA (seq+ss_join_*.txt)
SEQ_MIR_DIR = "./../data/sequences_miRNA"  # secuencias de miRNA  (seq+ss_join_*.txt)

OUT_LNC = "./../data/kmers/lnc_kmer_dict.txt"
OUT_MIR = "./../data/kmers/mir_kmer_dict.txt"

PAIRS_FILES = [
    "./../data/txt_interac/mirnas_lncrnas_validated_positive_pairs.txt",
    "./../data/txt_interac/negative_pairs.txt",
]

KS       = [1, 2, 3, 4]   # tamanos de k-mero a contar
ALPHABET = "ACGT"         # alfabeto de nucleotidos (la U se convierte a T)

# ONLY_PAIRS:
#   True  -> solo se generan los RNA que aparecen en los pares de interaccion
#            (rapido, archivo pequeno, suficiente para entrenar la red).
#   False -> se generan TODOS los archivos de las carpetas de secuencias
#            (lento; el archivo de lncRNA puede pesar ~1 GB).
ONLY_PAIRS = True

# NORMALIZE:
#   False -> conteos crudos de cada k-mero
#   True  -> frecuencias relativas (conteo / total de k-meros)
NORMALIZE = False

# MIRNA_REGION: la secuencia del miRNA en disco es el PRECURSOR.
#   "full"   -> usa el precursor completo
#   "mature" -> usa solo la region madura (letras en mayuscula)
MIRNA_REGION = "full"
# ============================================================

In [9]:
# Lista ordenada de todos los k-meros -> define el orden de las columnas del vector
KMER_LIST  = ["".join(p) for k in KS for p in itertools.product(ALPHABET, repeat=k)]
KMER_INDEX = {kmer: i for i, kmer in enumerate(KMER_LIST)}
DIM        = len(KMER_LIST)
print(f"Dimension del vector de k-meros: {DIM}  "
      f"({' + '.join(str(len(ALPHABET) ** k) for k in KS)})")


def read_sequence(path, mature_only=False):
    """Lee la linea 1 del archivo seq+ss_join_*.txt y la normaliza a alfabeto ACGT."""
    with open(path) as f:
        seq = f.readline().strip()
    if mature_only:                        # conserva solo la region en mayusculas
        seq = "".join(c for c in seq if c.isupper())
    return seq.upper().replace("U", "T")


def kmer_vector(seq):
    """Devuelve el vector de conteos (o frecuencias) de k-meros de una secuencia."""
    counts = [0] * DIM
    for k in KS:
        for i in range(len(seq) - k + 1):
            idx = KMER_INDEX.get(seq[i:i + k])
            if idx is not None:            # ignora k-meros con N u otros caracteres
                counts[idx] += 1
    if NORMALIZE:
        total = sum(counts) or 1
        return [c / total for c in counts]
    return counts

Dimension del vector de k-meros: 340  (4 + 16 + 64 + 256)


In [10]:
# Conjunto de RNA que realmente se usan en el entrenamiento (pares de interaccion)
needed_lnc, needed_mir = set(), set()
for pf in PAIRS_FILES:
    if os.path.exists(pf):
        for line in open(pf):
            parts = line.strip().split(",")
            if len(parts) >= 2:
                needed_lnc.add(parts[0])
                needed_mir.add(parts[1])
print(f"lncRNA en pares: {len(needed_lnc)}   |   miRNA en pares: {len(needed_mir)}")

lncRNA en pares: 3165   |   miRNA en pares: 267


In [11]:
PREFIX = "seq+ss_join_"


def generate_dict(seq_dir, out_path, keep=None, mature_only=False):
    """Recorre seq_dir, calcula el vector de k-meros de cada RNA y escribe out_path."""
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    written, skipped = 0, 0
    found_names = set()
    with open(out_path, "w") as out:
        for fname in sorted(os.listdir(seq_dir)):
            if not fname.startswith(PREFIX):
                continue
            name = os.path.splitext(fname[len(PREFIX):])[0]
            if keep is not None and name not in keep:
                skipped += 1
                continue
            seq = read_sequence(os.path.join(seq_dir, fname), mature_only)
            vec = kmer_vector(seq)
            out.write(name + "," + ",".join(str(v) for v in vec) + "\n")
            written += 1
            found_names.add(name)
    print(f"  -> {out_path}")
    print(f"     {written} entradas escritas, {skipped} archivos omitidos")
    return found_names

In [12]:
# --- Generar lnc_kmer_dict.txt ---
keep_lnc = needed_lnc if ONLY_PAIRS else None
print("Generando lnc_kmer_dict.txt ...")
found_lnc = generate_dict(SEQ_LNC_DIR, OUT_LNC, keep=keep_lnc)

Generando lnc_kmer_dict.txt ...
  -> ./../data/kmers/lnc_kmer_dict.txt
     3165 entradas escritas, 118084 archivos omitidos


In [13]:
# --- Generar mir_kmer_dict.txt ---
keep_mir = needed_mir if ONLY_PAIRS else None
print("Generando mir_kmer_dict.txt ...")
found_mir = generate_dict(SEQ_MIR_DIR, OUT_MIR, keep=keep_mir,
                          mature_only=(MIRNA_REGION == "mature"))

Generando mir_kmer_dict.txt ...
  -> ./../data/kmers/mir_kmer_dict.txt
     267 entradas escritas, 2389 archivos omitidos


In [14]:
# --- Verificacion de cobertura y formato ---
missing_lnc = needed_lnc - found_lnc
missing_mir = needed_mir - found_mir

print("=== VERIFICACION DE COBERTURA ===")
print(f"lncRNA de pares sin archivo de secuencia: {len(missing_lnc)}")
print(f"miRNA  de pares sin archivo de secuencia: {len(missing_mir)}")
if missing_lnc:
    print("  lncRNA faltantes (muestra):", sorted(missing_lnc)[:10])
if missing_mir:
    print("  miRNA  faltantes (muestra):", sorted(missing_mir)[:10])

print("\n=== FORMATO DE SALIDA ===")
with open(OUT_LNC) as f:
    line = f.readline().strip().split(",")
print(f"lnc_kmer_dict: nombre='{line[0]}', {len(line) - 1} valores (esperado {DIM})")
with open(OUT_MIR) as f:
    line = f.readline().strip().split(",")
print(f"mir_kmer_dict: nombre='{line[0]}', {len(line) - 1} valores (esperado {DIM})")

if not missing_lnc and not missing_mir:
    print("\nOK: todos los RNA de los pares tienen su vector de k-meros.")
else:
    print("\nATENCION: hay RNA sin secuencia. Los pares que los usen "
          "fallaran en generate_csv.ipynb (KeyError).")

=== VERIFICACION DE COBERTURA ===
lncRNA de pares sin archivo de secuencia: 0
miRNA  de pares sin archivo de secuencia: 0

=== FORMATO DE SALIDA ===
lnc_kmer_dict: nombre='NONHSAT000043.2', 340 valores (esperado 340)
mir_kmer_dict: nombre='hsa-let-7a-5p', 340 valores (esperado 340)

OK: todos los RNA de los pares tienen su vector de k-meros.
